<a href="https://colab.research.google.com/github/ghirr/Smart-Traffic-Flow-Prediction-and-Optimization/blob/main/Part4_Reinforcement_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partie 4 (bonus) : Reinforcement Learning, régulation d'un carrefour à feux

**Projet : Smart Traffic Flow Prediction and Optimization**

Les parties 2 et 3 **prédisent** le trafic. Ici, on s'intéresse à l'**optimisation** : comment **décider** (quand passer un feu au vert) pour réduire l'attente des véhicules ? C'est un problème de **décision séquentielle** : chaque décision change l'état futur (les files), ce qui est exactement le cadre du Reinforcement Learning.

**Le dataset ne contient ni feux ni décisions.** L'énoncé autorise un environnement **simulé** : on construit donc un simulateur de carrefour, mais on l'**alimente avec les flux réels** observés dans les cellules les plus chargées du réseau (export de la partie 1, `rl_flux_axes.parquet`). Les hypothèses du simulateur qui ne viennent pas des données sont indiquées clairement.

**Plan** : 1. données réelles et hypothèses · 2. environnement (états, actions, récompense) · 3. politiques de référence · 4. agent Q-learning · 5. évaluation sur scénarios réels et synthétiques · 6. analyse.

In [ ]:
import os, sys, json, time, warnings, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
SEED = 42
np.random.seed(SEED)

# ---- Données préparées par la partie 1 (dossier data_prepared/ du dépôt) ----
REPO_URL = "https://github.com/ghirr/Smart-Traffic-Flow-Prediction-and-Optimization.git"
BRANCH   = "main"                      # branche du dépôt qui contient data_prepared/ et results/
if not os.path.exists("data_prepared/meta.json") and "google.colab" in sys.modules:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
assert os.path.exists("data_prepared/meta.json"), "Dossier data_prepared/ introuvable : exécutez la partie 1 ou clonez le dépôt."
os.makedirs("results", exist_ok=True)

META   = json.load(open("data_prepared/meta.json"))
FEATS  = META["features"]
ETATS  = META["etats"]                 # ['congestionné', 'ralenti', 'fluide']
SEUILS = META["seuils_etat_m_s"]
SPLITS = ("train", "val", "test")

def charger(suffixe):
    """Retourne les trois jeux (train, val, test) : suffixe = 'raw' (arbres) ou 'scaled' (ANN)."""
    return [pd.read_parquet(f"data_prepared/{s}_{suffixe}.parquet") for s in SPLITS]

# Les trois tâches de prédiction (cibles de la minute suivante)
TASKS = {
    "etat":    dict(target="etat_next_code", kind="clf", nom="État du trafic"),
    "vitesse": dict(target="speed_next",     kind="reg", nom="Vitesse moyenne (m/s)"),
    "flux":    dict(target="n_veh_next",     kind="reg", nom="Flux (nb de véhicules)"),
}

from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             f1_score, recall_score, roc_auc_score, confusion_matrix)

def metriques(kind, y, pred, proba=None):
    """Métriques adaptées à la tâche : régression (MAE, RMSE, R²) ou classification (accuracy, F1 macro, rappels, AUC)."""
    if kind == "reg":
        return {"MAE": mean_absolute_error(y, pred), "RMSE": mean_squared_error(y, pred) ** 0.5, "R2": r2_score(y, pred)}
    rec = recall_score(y, pred, average=None, labels=[0, 1, 2])
    out = {"Accuracy": accuracy_score(y, pred), "F1_macro": f1_score(y, pred, average="macro"),
           "Rappel_congestionné": rec[0], "Rappel_ralenti": rec[1], "Rappel_fluide": rec[2]}
    if proba is not None:
        out["AUC_ovr"] = roc_auc_score(y, proba, multi_class="ovr")
    return out

def perte(kind, y, pred):
    """Critère unique de sélection (à minimiser) : MAE en régression, 1 - F1 macro en classification."""
    return mean_absolute_error(y, pred) if kind == "reg" else 1 - f1_score(y, pred, average="macro")

T1, T2, W = META["T1"], META["T2"], META["n_fenetres"]
flux = pd.read_parquet("data_prepared/rl_flux_axes.parquet")
CELLULES = sorted(flux.cell_id.unique())
A0 = flux.pivot(index="w", columns="cell_id", values="n_axe0").reindex(columns=CELLULES)      # véhicules / minute, axe 0
A1 = flux.pivot(index="w", columns="cell_id", values="n_axe1").reindex(columns=CELLULES)      # véhicules / minute, axe 1
print(f"{len(CELLULES)} cellules, {len(A0)} minutes. Train : minutes < {T1}, validation : {T1}-{T2}, test : > {T2}")
flux.groupby("cell_id")[["n_axe0", "n_axe1"]].agg(["mean", lambda s: s.quantile(.95)]).round(1).set_axis(
    ["axe0 moyenne", "axe0 pic (p95)", "axe1 moyenne", "axe1 pic (p95)"], axis=1)

## 1. Données réelles et hypothèses

**Ce qui vient des données.** Pour les 5 cellules les plus chargées, le nombre de véhicules par minute sur chacun des **deux axes** de circulation (les deux familles de routes du damier). Ces cellules sont d'ailleurs lentes : la vitesse moyenne y est de 0,7 à 2,4 m/s, ce sont de vrais points de congestion. On les traite comme des **carrefours** à deux phases.

**Ce qui est supposé** (le dataset ne le contient pas) :

| Hypothèse | Valeur | Justification |
|---|---|---|
| Flux de saturation (véhicules écoulés par seconde au vert) | 0,5 véh/s (1 800 véh/h) | valeur classique d'une voie urbaine |
| Temps perdu à chaque changement de phase (orange + dégagement) | 3 s | ordre de grandeur usuel |
| Vert minimal | 10 s | sécurité / confort |
| Arrivées | loi de Poisson dont la moyenne suit les flux réels par minute | processus d'arrivée standard |

**Découpage** : l'agent apprend sur les minutes du **train**, les réglages (pas de temps du feu fixe, paramètres de l'heuristique, choix de la meilleure politique apprise) se font sur la **validation**, et l'évaluation finale utilise les minutes du **test** (période de simulation postérieure, jamais vue).

## 2. L'environnement : états, actions, récompense

| Élément | Définition |
|---|---|
| **Carrefour** | 2 axes (0 et 1) ; un seul axe est au vert à la fois |
| **État observé** | longueur de la file de chaque axe (discrétisée en 8 classes), phase en cours (quel axe est au vert), temps écoulé depuis le dernier changement (3 classes) : 8 × 8 × 2 × 3 = **384 états** |
| **Actions** | `0` = garder la phase ; `1` = changer de phase (refusé si le vert minimal n'est pas écoulé) |
| **Pas de décision** | 5 secondes (la simulation avance seconde par seconde) |
| **Récompense** | `−(nombre de véhicules en attente, cumulé sur le pas) / 50` : minimiser l'attente totale revient à minimiser le retard moyen (loi de Little) |
| **Épisode** | 15 minutes (900 s), soit 180 décisions, avec les flux réels de 15 minutes consécutives |
| **Dynamique** | arrivées de Poisson ; à chaque seconde, si l'axe au vert a une file non vide et hors temps perdu, un véhicule est écoulé avec la probabilité 0,5 |

In [ ]:
class Carrefour:
    """Carrefour à 2 phases. Simulation à la seconde, décision toutes les DT secondes."""
    DT, MIN_GREEN, LOST, QMAX = 5, 10, 3, 80
    BORNES = [0, 1, 2, 4, 7, 11, 17, 26]                    # 8 classes de longueur de file
    N_ETATS, N_ACTIONS = 8 * 8 * 2 * 3, 2

    def __init__(self, saturation=0.5):
        self.S = saturation

    def reset(self, lam0, lam1, seed, horizon=900):
        """lam0, lam1 : arrivées (véhicules / minute), une valeur par minute de l'épisode."""
        rng = np.random.default_rng(seed)
        minute = np.arange(horizon) // 60
        self.arr = [rng.poisson(np.asarray(lam0, float)[minute] / 60), rng.poisson(np.asarray(lam1, float)[minute] / 60)]
        self.srv = rng.random(horizon) < self.S
        self.H, self.t = horizon, 0
        self.q, self.phase, self.tp, self.lost = [0, 0], 0, self.MIN_GREEN, 0
        self.wait = self.served = self.arrived = self.switches = self.qmax = 0
        return self.etat()

    def etat(self):
        tb = 0 if self.tp < self.MIN_GREEN else (1 if self.tp < 25 else 2)
        return ((self.CLASSE[self.q[0]] * 8 + self.CLASSE[self.q[1]]) * 2 + self.phase) * 3 + tb

    def step(self, action):
        if action == 1 and self.tp >= self.MIN_GREEN:        # changement de phase : temps perdu, plus personne ne passe
            self.phase ^= 1; self.tp = 0; self.lost = self.LOST; self.switches += 1
        somme = 0
        for _ in range(self.DT):
            t = self.t
            for k in (0, 1):
                a = int(self.arr[k][t]); self.arrived += a
                self.q[k] = min(self.QMAX, self.q[k] + a)
            if self.lost > 0:
                self.lost -= 1
            elif self.q[self.phase] > 0 and self.srv[t]:
                self.q[self.phase] -= 1; self.served += 1
            self.tp += 1; self.t += 1
            tot = self.q[0] + self.q[1]; somme += tot; self.qmax = max(self.qmax, tot)
        self.wait += somme
        return self.etat(), -somme / 50.0, self.t >= self.H

    def bilan(self):
        return {"retard_s_par_veh": self.wait / max(1, self.arrived), "file_moyenne": self.wait / self.H,
                "debit_veh_h": self.served / self.H * 3600, "file_max": self.qmax, "changements_h": self.switches / self.H * 3600}

Carrefour.CLASSE = [int(np.searchsorted(Carrefour.BORNES, q, side="right") - 1) for q in range(Carrefour.QMAX + 1)]   # longueur de file -> classe

## 3. Politiques de référence

Un agent appris n'a de valeur que s'il bat des **règles simples**. On en teste quatre :

- **Aléatoire** : change de phase au hasard (borne basse).
- **Temps fixe** : change de phase toutes les `G` secondes, quel que soit le trafic (c'est ce que font beaucoup de feux réels). On teste `G = 30 s` puis la meilleure valeur sur la validation.
- **Plus longue file** : change de phase quand la file de l'axe en rouge dépasse celle de l'axe vert d'une marge `m` (ou quand le vert est vide), avec la marge `m` choisie sur la validation. C'est une heuristique **réactive**, souvent très difficile à battre.

In [ ]:
def pol_aleatoire(seed=0):
    rng = np.random.default_rng(seed)
    return lambda env: int(rng.random() < 0.1)

def pol_fixe(G):
    return lambda env: int(env.tp >= G)

def pol_longue_file(marge):
    def pol(env):
        v, r = env.q[env.phase], env.q[1 - env.phase]
        return int(env.tp >= env.MIN_GREEN and (r > v + marge or (v == 0 and r > 0)))
    return pol

def jouer(politique, lam0, lam1, seed, saturation=0.5):
    env = Carrefour(saturation); env.reset(lam0, lam1, seed)
    done = False
    while not done:
        _, _, done = env.step(politique(env))
    return env.bilan()

## 4. Agent : Q-learning tabulaire

**Principe** (guide Q-learning du cours). L'agent apprend une table `Q(état, action)` = « récompense future attendue si je fais cette action dans cet état ». À chaque transition `(s, a, r, s')` :

`Q(s, a) ← Q(s, a) + α · ( r + γ · max Q(s', ·) − Q(s, a) )`

- **ε-greedy** : avec une probabilité `ε` on explore (action au hasard), sinon on exploite (meilleure action connue). `ε` décroît de 1 à 0,05.
- `α` = taux d'apprentissage (décroît de 0,2 à 0,03), `γ` = 0,97 (les conséquences d'un changement de phase se voient sur plusieurs décisions).

**Pourquoi Q-learning ?** L'espace d'états est **petit et discret** (384 états), l'algorithme est celui vu en cours, et la table apprise est **lisible** (on peut visualiser la règle de décision). Ses limites (pas de généralisation entre états, ne passe pas à l'échelle d'un réseau entier) sont discutées dans l'analyse.

**Entraînement** : à chaque épisode on tire au hasard une des 5 cellules et un créneau de 15 minutes **dans la période de train** ; l'agent subit donc des profils de demande réels et variés. Toutes les 400 épisodes on évalue la politique gloutonne sur la **validation** et on garde la meilleure table.

In [ ]:
DUREE = 15                                                  # minutes par épisode

def segment(periode, cell, rng):
    lo, hi = {"train": (10, T1), "val": (T1 + 1, T2), "test": (T2 + 1, W - 1)}[periode]
    w0 = int(rng.integers(lo, hi - DUREE + 1))
    return A0[cell].values[w0:w0 + DUREE], A1[cell].values[w0:w0 + DUREE], w0

def scenarios_reels(periode):
    """Créneaux consécutifs de 15 minutes de chaque cellule, dans la période choisie."""
    lo, hi = {"val": (T1 + 1, T2), "test": (T2 + 1, W - 1)}[periode]
    out = []
    for c in CELLULES:
        for w0 in range(lo, hi - DUREE + 1, DUREE):
            out.append({"nom": f"cellule {c} / min {w0}", "cell": c, "lam0": A0[c].values[w0:w0 + DUREE], "lam1": A1[c].values[w0:w0 + DUREE]})
    return out

def evaluer(politique_factory, scens, n_seeds=4, saturation=0.5):
    """Retard moyen sur tous les scénarios x graines (mêmes aléas pour toutes les politiques : comparaison appariée)."""
    res = []
    for i, s in enumerate(scens):
        for k in range(n_seeds):
            res.append({"scenario": s["nom"], "graine": k, **jouer(politique_factory(), s["lam0"], s["lam1"], 1000 * i + k, saturation)})
    return pd.DataFrame(res)

def entrainer_q(n_episodes=30000, seed=SEED, alpha=(0.1, 0.01), gamma=0.97, eps=(1.0, 0.05), verbose=True):
    rng = np.random.default_rng(seed)
    Q = np.zeros((Carrefour.N_ETATS, Carrefour.N_ACTIONS)); visites = np.zeros(Carrefour.N_ETATS)
    env = Carrefour()
    val = scenarios_reels("val")
    meilleur, Q_best, journal, journal_val = np.inf, Q.copy(), [], []
    for ep in range(n_episodes):
        frac = ep / n_episodes
        e = max(eps[1], eps[0] - (eps[0] - eps[1]) * frac / 0.7)
        a_ = alpha[0] + (alpha[1] - alpha[0]) * frac
        cell = CELLULES[int(rng.integers(len(CELLULES)))]
        l0, l1, _ = segment("train", cell, rng)
        s = env.reset(l0, l1, int(rng.integers(1 << 30)))
        retour, done = 0.0, False
        while not done:
            a = int(rng.integers(2)) if rng.random() < e else int(Q[s].argmax())
            s2, r, done = env.step(a)
            visites[s] += 1
            Q[s, a] += a_ * (r + (0 if done else gamma * Q[s2].max()) - Q[s, a])
            s, retour = s2, retour + r
        journal.append(retour)
        if (ep + 1) % 1000 == 0:
            q_pol = lambda env_, Q=Q: int(Q[env_.etat()].argmax())
            d = evaluer(lambda: q_pol, val, n_seeds=3).retard_s_par_veh.mean()
            journal_val.append((ep + 1, d))
            if d < meilleur: meilleur, Q_best = d, Q.copy()
            if verbose and (ep + 1) % 5000 == 0: print(f"  épisode {ep + 1:5d} | ε = {e:.2f} | retard validation = {d:.2f} s/véh (meilleur : {meilleur:.2f})")
    return Q_best, np.array(journal), journal_val, meilleur, visites

# Le Q-learning dépend du hasard (exploration, tirage des épisodes) : on entraîne 5 agents de graines différentes
# et on garde celui qui a le meilleur retard sur la VALIDATION (le test reste intact).
AGENTS = []
t0 = time.time()
for sd in range(5):
    Qs, ret_s, jv_s, score_val, vis = entrainer_q(seed=SEED + sd, verbose=(sd == 0))
    AGENTS.append({"seed": SEED + sd, "Q": Qs, "retours": ret_s, "journal_val": jv_s, "retard_val": score_val, "visites": vis})
    print(f"agent {sd} (graine {SEED + sd}) : meilleur retard validation = {score_val:.2f} s/véh")
print(f"Entraînement terminé en {time.time() - t0:.0f} s")
CHOIX = int(np.argmin([a["retard_val"] for a in AGENTS]))
Q, retours, journal_val, VISITES = AGENTS[CHOIX]["Q"], AGENTS[CHOIX]["retours"], AGENTS[CHOIX]["journal_val"], AGENTS[CHOIX]["visites"]
print(f"Agent retenu (validation) : n°{CHOIX}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
r = pd.Series(retours).rolling(200).mean()
ax[0].plot(r); ax[0].set_xlabel("épisode"); ax[0].set_ylabel("retour (moyenne glissante 200)"); ax[0].set_title("Entraînement : récompense cumulée par épisode\n(avec exploration, donc inférieure à la politique gloutonne)")
for a_ in AGENTS:
    jv = pd.DataFrame(a_["journal_val"], columns=["episode", "retard"]); ax[1].plot(jv.episode, jv.retard, alpha=0.8, label=f"graine {a_['seed']}")
ax[1].legend(fontsize=7); ax[1].set_xlabel("épisode"); ax[1].set_ylabel("retard moyen validation (s/véh)")
ax[1].set_title("Validation (politique gloutonne)")
plt.tight_layout(); plt.show()

## 5. Évaluation

### 5.1 Réglage des références sur la validation

In [ ]:
val = scenarios_reels("val")
grille_G = [10, 15, 20, 30, 45, 60]
ev = {G: evaluer(lambda G=G: pol_fixe(G), val, 2).retard_s_par_veh.mean() for G in grille_G}
G_OPT = min(ev, key=ev.get)
ev_m = {m: evaluer(lambda m=m: pol_longue_file(m), val, 2).retard_s_par_veh.mean() for m in [0, 1, 2, 4, 6, 8, 10, 14]}
M_OPT = min(ev_m, key=ev_m.get)
print("Temps fixe, retard moyen (validation) selon G :", {k: round(v, 1) for k, v in ev.items()}, f"-> G = {G_OPT} s")
print("Plus longue file, retard moyen selon la marge  :", {k: round(v, 1) for k, v in ev_m.items()}, f"-> marge = {M_OPT}")

q_pol = lambda env, Q=Q: int(Q[env.etat()].argmax())
POLITIQUES = {"Aléatoire": lambda: pol_aleatoire(1),
              f"Temps fixe ({G_OPT} s, optimisé)": lambda: pol_fixe(G_OPT), f"Plus longue file (marge {M_OPT})": lambda: pol_longue_file(M_OPT),
              "Q-learning": lambda: q_pol}

### 5.2 Scénarios **réels** de test

Créneaux de 15 minutes pris dans la période de **test** (la fin de la simulation, jamais vue à l'entraînement), pour chacune des 5 cellules.

In [ ]:
test_reel = scenarios_reels("test")
res_reel = {nom: evaluer(f, test_reel, 6) for nom, f in POLITIQUES.items()}
tab = pd.DataFrame({nom: d[["retard_s_par_veh", "file_moyenne", "debit_veh_h", "file_max", "changements_h"]].mean() for nom, d in res_reel.items()}).T
tab["IC95 retard (±)"] = [1.96 * d.retard_s_par_veh.std() / np.sqrt(len(d)) for d in res_reel.values()]
print(f"{len(test_reel)} créneaux x 6 graines = {len(test_reel) * 6} épisodes par politique\n")
print(tab.round(2).to_string())

In [ ]:
# Variabilité du Q-learning : les 5 agents sur les mêmes scénarios de test
agents_test = []
for a_ in AGENTS:
    pol = (lambda env, Qa=a_["Q"]: int(Qa[env.etat()].argmax()))
    agents_test.append(float(evaluer(lambda: pol, test_reel, 6).retard_s_par_veh.mean()))
print("Retard moyen sur le test, par agent (s/véh) :", [round(v, 2) for v in agents_test])
print(f"Moyenne des 5 agents = {np.mean(agents_test):.2f}, écart-type = {np.std(agents_test):.2f}, agent retenu sur la validation : {agents_test[CHOIX]:.2f}")

In [ ]:
# Comparaison appariée Q-learning vs plus longue file : mêmes créneaux, mêmes aléas, différence épisode par épisode
nom_h = [n for n in res_reel if n.startswith("Plus longue")][0]
diff = (res_reel["Q-learning"].retard_s_par_veh.values - res_reel[nom_h].retard_s_par_veh.values)
ic = 1.96 * diff.std() / np.sqrt(len(diff))
print(f"Δ retard (Q-learning - plus longue file) = {diff.mean():+.2f} s/véh, IC95 [{diff.mean() - ic:+.2f}, {diff.mean() + ic:+.2f}] sur {len(diff)} épisodes appariés")
print("Épisodes où Q-learning fait mieux :", f"{100 * (diff < 0).mean():.0f} %")

### 5.3 Scénarios synthétiques de stress

Pour tester la **robustesse** de l'agent face à des situations **absentes de l'entraînement** (le train ne contient que des profils réels). Flux en véhicules/minute pour chaque axe, sur 15 minutes :

| Scénario | Description |
|---|---|
| Équilibré, modéré | 10 / 10 |
| Asymétrique | 4 / 18 (un axe très dominant) |
| Forte charge | 18 / 18 (proche de la saturation) |
| Pic soudain | 6 / 6 pendant 5 min, puis 20 / 20 pendant 5 min, puis 6 / 6 |
| Capacité réduite | 12 / 12 mais flux de saturation 0,4 au lieu de 0,5 (pluie, incident) |

In [ ]:
def const(a, b): return {"lam0": [a] * DUREE, "lam1": [b] * DUREE}
SYNTH = {
    "Équilibré, modéré": dict(nom="eq", **const(10, 10), S=0.5),
    "Asymétrique":       dict(nom="asym", **const(4, 18), S=0.5),
    "Forte charge":      dict(nom="forte", **const(18, 18), S=0.5),
    "Pic soudain":       dict(nom="pic", lam0=[6] * 5 + [20] * 5 + [6] * 5, lam1=[6] * 5 + [20] * 5 + [6] * 5, S=0.5),
    "Capacité réduite":  dict(nom="cap", **const(12, 12), S=0.4),
}
res_syn = {}
for scen, cfg in SYNTH.items():
    res_syn[scen] = {nom: evaluer(f, [cfg], 20, saturation=cfg["S"]).retard_s_par_veh.mean() for nom, f in POLITIQUES.items()}
syn = pd.DataFrame(res_syn).T
print("Retard moyen par véhicule (secondes), 20 graines par scénario :")
print(syn.round(1).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
tab["retard_s_par_veh"].plot.barh(ax=ax[0], xerr=tab["IC95 retard (±)"], color=["gray", "steelblue", "seagreen", "darkorange"])
ax[0].set_xlabel("retard moyen (s / véhicule)"); ax[0].set_title("Scénarios réels (test) : plus bas = mieux")
syn.plot.bar(ax=ax[1], width=0.85); ax[1].set_ylabel("retard moyen (s / véhicule)"); ax[1].set_title("Scénarios synthétiques de stress")
ax[1].legend(fontsize=7); plt.setp(ax[1].get_xticklabels(), rotation=20, ha="right")
plt.tight_layout(); plt.show()

### 5.4 Comportement de l'agent

**Que fait la politique apprise ?** On visualise l'action choisie selon les files des deux axes quand l'axe 0 est au vert (vert minimal écoulé) : en bleu, l'agent **change** de phase ; en blanc, il **garde**.

In [ ]:
import matplotlib.colors as mcolors
etiq = ["0", "1", "2-3", "4-6", "7-10", "11-16", "17-25", "26+"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for a, phase in zip(ax, [0, 1]):
    M = np.array([[int(Q[((i * 8 + j) * 2 + phase) * 3 + 2].argmax()) for j in range(8)] for i in range(8)])
    visite = np.array([[VISITES[((i * 8 + j) * 2 + phase) * 3 + 2] >= 200 for j in range(8)] for i in range(8)])      # états rencontrés au moins 200 fois
    sns.heatmap(np.where(visite, M, np.nan), cmap=mcolors.ListedColormap(["white", "steelblue"]), cbar=False, ax=a, linewidths=.5,
                xticklabels=etiq, yticklabels=etiq, linecolor="lightgray", annot=np.where(visite, np.where(M == 1, "change", "garde"), ""), fmt="s", annot_kws={"fontsize": 7})
    a.set_xlabel("file de l'axe 1"); a.set_ylabel("file de l'axe 0"); a.set_title(f"Phase {phase} : l'axe {phase} est au vert (vert > 25 s)")
plt.tight_layout(); plt.show()
print(f"Cases vides = états rares (moins de 200 passages pendant l'entraînement) : la décision y est peu fiable. États visités au moins 200 fois : {int((VISITES >= 200).sum())} sur {Carrefour.N_ETATS}")

In [ ]:
# Un épisode du test : files au cours du temps, temps fixe vs Q-learning
sc = test_reel[0]
def trace(politique, seed=0):
    env = Carrefour(); env.reset(sc["lam0"], sc["lam1"], seed); h, done = [], False
    while not done:
        _, _, done = env.step(politique(env)); h.append((env.t, env.q[0], env.q[1], env.phase))
    return pd.DataFrame(h, columns=["t", "q0", "q1", "phase"])
fig, ax = plt.subplots(1, 2, figsize=(14, 3.4), sharey=True)
for a, (nom, f) in zip(ax, [(f"Temps fixe ({G_OPT} s)", pol_fixe(G_OPT)), ("Q-learning", q_pol)]):
    d = trace(f); a.plot(d.t, d.q0, label="file axe 0"); a.plot(d.t, d.q1, label="file axe 1")
    a.set_title(f"{nom} ({sc['nom']})"); a.set_xlabel("temps (s)"); a.legend()
ax[0].set_ylabel("véhicules en attente")
plt.tight_layout(); plt.show()

In [ ]:
# Sauvegarde pour la partie 5
reel = {nom: {"retard_s_par_veh": float(d.retard_s_par_veh.mean()), "ic95": float(1.96 * d.retard_s_par_veh.std() / np.sqrt(len(d))),
              "file_moyenne": float(d.file_moyenne.mean()), "debit_veh_h": float(d.debit_veh_h.mean()),
              "file_max": float(d.file_max.mean()), "changements_h": float(d.changements_h.mean())} for nom, d in res_reel.items()}
json.dump({"reel_test": reel, "synthetique": {k: {n: float(v) for n, v in r.items()} for k, r in res_syn.items()},
           "parametres": {"G_optimal": int(G_OPT), "marge_optimale": int(M_OPT), "episodes": 30000, "agents_q_test": agents_test,
                          "retard_par_scenario_reel": {nom: d.groupby("scenario").retard_s_par_veh.mean().to_dict() for nom, d in res_reel.items()}}},
          open("results/rl_metrics.json", "w"), indent=2, ensure_ascii=False)
np.save("results/rl_qtable.npy", Q)
print("Sauvegardé : results/rl_metrics.json et results/rl_qtable.npy")

## 6. Analyse

### Ce que montre la comparaison
- **Cadre** : on compare quatre politiques sur 15 scénarios construits à partir des flux réels de la période de test (5 cellules parmi les plus chargées × 3 instants), avec des arrivées simulées identiques pour toutes les politiques (nombres aléatoires communs) et un test apparié.
- **Retard moyen par véhicule** : aléatoire 68,9 s ; temps fixe optimisé (30 s) 30,1 s ; plus longue file 11,9 s ; **Q-learning 11,5 s**. L'agent réduit donc le retard d'environ **62 %** par rapport au temps fixe et de **83 %** par rapport à une politique aléatoire.
- **Face à l'heuristique « plus longue file »** (réglée sur la validation, donc pas un adversaire facile) : le gain moyen est faible, de **−0,34 s** par véhicule, mais l'intervalle de confiance à 95 % de la différence appariée exclut 0. L'agent est donc *légèrement* meilleur, pas radicalement. Les 5 agents entraînés avec des graines différentes donnent des retards de 11,5 à 12,0 s : la dispersion est du même ordre que le gain, il faut donc rester prudent.
- **Scénarios synthétiques** : le temps fixe s'effondre quand le trafic est asymétrique (109,6 s contre 12,1 s pour l'agent) puisqu'il ne s'adapte pas ; l'agent et l'heuristique sont à égalité quand le trafic est équilibré ou la capacité réduite (49,9 s contre 49,7 s) ; l'avantage de l'agent apparaît surtout en **forte charge** (93,0 s contre 111,3 s, soit −16 %) et face à un **pic soudain** (55,4 s contre 59,0 s). Quand les files saturent, la règle « servir la plus longue file » n'est plus optimale, alors que l'agent a appris à anticiper.
- **Politique apprise** (carte ci-dessus, vert > 25 s) : la structure principale est celle attendue. L'agent **change de phase quand la file de l'axe au vert est vide (ou presque) alors que l'autre axe attend**, et il **garde** la phase tant que l'axe servi a encore des véhicules. On retrouve donc l'esprit de la règle « plus longue file ». La carte n'est pourtant pas aussi régulière que la règle écrite à la main : quelques cases sont irrégulières (par exemple « change » pour une file de 17-25 mais « garde » pour 26+ sur l'autre axe), ce qui indique que, dans ces états plus rares, la valeur apprise reste bruitée. C'est une limite de l'apprentissage tabulaire par exploration, qui explique en partie pourquoi le gain sur l'heuristique est faible.

### Limites (à garder à l'esprit)
- L'environnement est une **simulation simplifiée** d'un carrefour à 2 phases, alimentée par les flux réels agrégés par minute et par axe ; les hypothèses (saturation 0,5 véh/s, temps perdu 3 s, vert minimal 10 s) sont les nôtres, pas celles du simulateur d'origine. Les chiffres de retard sont donc **relatifs** (comparaison entre politiques), pas des retards réels du réseau.
- Un Q-learning **tabulaire** exige de discrétiser l'état (files, phase, durée de vert) ; la précision est bornée par cette discrétisation.
- Le gain sur l'heuristique est modeste ; son intérêt est d'apprendre sans règle écrite à la main, et de mieux gérer les situations saturées.